# do-Shapley Quick Validation (Tabular Data)

Quick validation of the `DoCausalImputer` bug fixes on the Bike Sharing dataset (11 features, 587 train samples).  
Tests Marginal SHAP / DML / IPW / REG on 3 test samples. **Runtime: ~50 minutes.**

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
import time as time_module

import shapiq
from shapiq import TabularExplainer
from shapiq.imputer import DoCausalImputer
from shapiq.datasets import load_bike_sharing_daily, load_bike_sharing_daily_train_index

print(f"shapiq version: {shapiq.__version__}")

In [ ]:
# Load data
x_preprocessed, y_data = load_bike_sharing_daily(preprocessed=True)
x_raw, _ = load_bike_sharing_daily(preprocessed=False)
train_idx = load_bike_sharing_daily_train_index()

x_data_raw = x_preprocessed.copy()
x_data_raw['weekday'] = x_raw['weekday'].values
x_data_raw['workingday'] = x_raw['workingday'].values
x_data_raw['holiday'] = x_raw['holiday'].values
x_data_raw['weather'] = x_raw['weathersit'].values
x_data_raw['location'] = 1.0
x_data_raw.rename(columns={'hum': 'humidity', 'atemp': 'feel_temp'}, inplace=True)

feature_names = [
    "trend", "sinyear", "weekday", "location", "workingday",
    "holiday", "weather", "temp", "humidity", "windspeed", "feel_temp",
]
x_data = x_data_raw[feature_names].copy()

all_idx = np.arange(len(x_data))
test_idx = np.array([i for i in all_idx if i not in train_idx])

X_train = x_data.iloc[train_idx].values
y_train_nc = y_data.iloc[train_idx].values
y_train_mean = y_train_nc.mean()
y_train = y_train_nc - y_train_mean

X_test = x_data.iloc[test_idx].values
y_test = y_data.iloc[test_idx].values - y_train_mean
n_features = len(feature_names)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# Train model
model = xgb.XGBRegressor(n_estimators=100, random_state=1, verbosity=0)
model.fit(X_train, y_train)
rmse = np.sqrt(np.mean((y_test - model.predict(X_test)) ** 2))
print(f"Test RMSE: {rmse:.2f}")

In [ ]:
# DAG
dag_edges = [
    (0, 4), (0, 5), (2, 4), (2, 5), (3, 4), (3, 5),
    (0, 1), (0, 6), (0, 7), (0, 8), (0, 9),
    (3, 6), (3, 7), (3, 8), (3, 9),
    (6, 10), (7, 10), (8, 10), (9, 10),
]
confounding_pairs = DoCausalImputer.confounding_group(6, 7, 8, 9)

In [ ]:
# Create explainers
N_QUICK = 3
budget = 2048
RANDOM_STATE = 42

explainers = {}
explainers['Marginal SHAP'] = TabularExplainer(
    model=model.predict, data=X_train, imputer="marginal",
    index="SV", max_order=1, random_state=RANDOM_STATE
)
for method in ['dml', 'ipw', 'reg']:
    explainers[f'do-Shapley ({method.upper()})'] = TabularExplainer(
        model=model.predict, data=X_train, imputer="do_causal",
        index="SV", max_order=1, random_state=RANDOM_STATE,
        dag_edges=dag_edges, confounding_pairs=confounding_pairs, method=method,
    )

# Verify conditioning sets
imp = explainers['do-Shapley (DML)'].imputer
print(f"cross_fitting: {imp.cross_fitting}")
for vi in imp.topo_order:
    mi = imp._cond_prob_models[vi]
    if mi['type'] == 'marginal':
        print(f"  {feature_names[vi]}: marginal")
    else:
        print(f"  {feature_names[vi]}: P(· | {[feature_names[p] for p in mi['pre_indices']]})")

In [ ]:
# Compute Shapley values (~50 min total)
sv = {name: np.zeros((N_QUICK, n_features)) for name in explainers}

t_total = time_module.time()
for name, exp in explainers.items():
    t0 = time_module.time()
    for i in range(N_QUICK):
        result = exp.explain(X_test[i:i+1], budget=budget)
        sv[name][i] = result.get_n_order_values(order=1)
    print(f"  {name}: {time_module.time()-t0:.1f}s")
print(f"Total: {time_module.time()-t_total:.1f}s")

In [ ]:
# Results
print(f"{'Feature':>12s} {'Marginal':>10s} {'DML':>10s} {'IPW':>10s} {'REG':>10s}")
print(f"{'-'*12} {'-'*10} {'-'*10} {'-'*10} {'-'*10}")
for f_idx, fname in enumerate(feature_names):
    vals = [np.abs(sv[n][:, f_idx]).mean() for n in explainers]
    print(f"{fname:>12s} {vals[0]:>10.1f} {vals[1]:>10.1f} {vals[2]:>10.1f} {vals[3]:>10.1f}")

# location diagnostic
loc_idx = feature_names.index('location')
print(f"\nlocation (const=1) importance:")
for name in explainers:
    v = np.abs(sv[name][:, loc_idx]).mean()
    print(f"  {name}: {v:.1f}  {'✗' if v > 200 else '✓'}")

# temp/sinyear ratio
temp_idx, sin_idx = feature_names.index('temp'), feature_names.index('sinyear')
print(f"\ntemp/sinyear ratio:")
for name in explainers:
    t = np.abs(sv[name][:, temp_idx]).mean()
    s = np.abs(sv[name][:, sin_idx]).mean()
    print(f"  {name}: {t/s:.2f}")